
# DOPe-Bench: Experiment Pipeline

This notebook implements an experiment suite for **DOPe** (Danger-Obsessed Pedestrians),
built around what is actually annotated in `1-video_annotations_dataset.csv` (1,027 raw
scenarios / 910 curated, 84 source videos, 4 tag families) rather than the full aspirational
scope of the "master specification" draft.

## Why the scope here is narrower than the master-spec doc

The master-spec document proposes `DOPe-Risk` (continuous risk scores from TTC / relative
velocity / distance), `DOPe-Intervene` (counterfactual agent/action/timing labels), and a
true `DOPe-Early` (per-timestep risk trajectories anchored to a labeled "critical event"
timestamp). **None of those quantities exist in the current CSV.** The CSV gives you, per
scenario: a `start_frame`/`end_frame` interval and four label sets (pedestrian behavior,
vehicle response, environment, archetype). There is no distance/velocity/TTC channel, no
counterfactual annotation, and no sub-event timestamp inside a scenario to anchor an
early-warning curve to.

So this notebook builds the two things the data actually supports well, plus two
**proxy** versions of the stress tests that are honest about being proxies:

| Task | Status | What it uses |
|---|---|---|
| **DOPe-Understand** (multi-label recognition) | Fully supported — this is your current draft's Section IV | video clips + 4 tag families |
| **Dataset analysis** (Tables II-V: archetype coverage, behavior/outcome profiles, overlaps, occlusion subset) | Fully supported, no VLM needed | CSV only |
| **DOPe-Compose-lite** (compositional generalization) | Proxy — defined over *tag co-occurrence* splits, not risk-factor splits | CSV + clips |
| **DOPe-Early-lite** (context-truncation degradation) | Proxy — truncates the annotated interval itself as a stand-in for "less pre-event context"; there's no independent critical-event timestamp to anchor a real early-warning curve | clips only |
| **Table I** (inter-annotator κ) | **Cannot be reproduced from this file** | needs the two pre-adjudication annotation passes, not the merged/agreed CSV |
| **DOPe-Risk / DOPe-Intervene** (from master spec) | **Not attempted here** | needs new annotation: TTC/distance/velocity, counterfactual action labels |

If you want the full master-spec scope, the fastest path is annotating a subset (even
150-200 scenarios) with TTC/distance proxies (can be semi-automated with a tracker + camera
calibration) and counterfactual action/timing labels — flagged in the Limitations cell at
the end with a concrete annotation-effort estimate.

## Compute model

Per your compute-resources preference (CPU-accessible pipeline, GPU only where unavoidable):
- **GPT-4o / GPT-4o-mini / Claude-Sonnet / Gemini-2.5-Flash**: API calls, CPU-only, no GPU needed.
- **Qwen3-VL-32B**: the only open-weight model in the baseline comparison — needs a GPU.
  A Modal deployment stub is included (you mentioned you have Modal GPU credits) so this
  is the only part of the pipeline that leaves your laptop/CPU box.
- Video download (`yt-dlp`) + frame extraction (`ffmpeg`/`opencv`) + all dataset-statistics
  and metrics code: CPU-only.

## GPU requirements (summary — details in the Qwen3-VL section below)

| Model | Precision | Min VRAM | Recommended | Notes |
|---|---|---|---|---|
| Qwen3-VL-32B | bf16 | ~66 GB | 1x A100-80GB or 1x H100-80GB | video frames add activation memory on top of weights; budget headroom |
| Qwen3-VL-32B | 4-bit (bitsandbytes/AWQ) | ~20-22 GB | 1x A100-40GB or 1x L40S-48GB | slower, ~2-4% accuracy drop typical for 4-bit VLMs |
| Everything else | n/a | 0 (API) | CPU instance is fine | rate-limit bound, not compute bound |

If you run the full 910-scenario x 5-model x (Understand + Compose + Early) sweep on
Qwen3-VL-32B at ~8 frames/scenario, that's roughly 910 x 3 x 8 ≈ 21,800 forward passes worth
of frames per full pass — budget Modal GPU time accordingly (see cost-estimate cell).


## 0. Environment setup

In [ ]:

# Core dataset / stats stack (CPU only)
!pip install -q pandas numpy scikit-learn matplotlib seaborn rapidfuzz tqdm

# Video acquisition + frame extraction (CPU only)
!pip install -q yt-dlp opencv-python-headless

# API clients for the proprietary models (CPU only — network calls)
!pip install -q openai anthropic google-generativeai

# Only needed if you run Qwen3-VL-32B locally instead of via Modal
# !pip install -q transformers accelerate qwen-vl-utils bitsandbytes


In [ ]:

import os
import re
import json
import math
import time
import random
import subprocess
from pathlib import Path
from dataclasses import dataclass, field
from collections import Counter, defaultdict

import pandas as pd
import numpy as np
from rapidfuzz import fuzz, process as rf_process
from tqdm.auto import tqdm

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ---- Paths ----------------------------------------------------------------
DATA_CSV      = Path("1-video_annotations_dataset.csv")   # the file you uploaded
VIDEO_DIR     = Path("data/videos")        # full downloaded source videos
CLIP_DIR      = Path("data/clips")         # trimmed per-scenario clips
FRAME_DIR     = Path("data/frames")        # extracted frames per scenario
RESULTS_DIR   = Path("results")
for d in (VIDEO_DIR, CLIP_DIR, FRAME_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---- API keys (set these as environment variables, do not hardcode) -------
OPENAI_API_KEY    = os.environ.get("OPENAI_API_KEY")
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
GOOGLE_API_KEY    = os.environ.get("GOOGLE_API_KEY")

# ---- Frame sampling policy (matches Fig. 1 in your draft) ------------------
MIN_FRAMES = 4
SAMPLE_FPS = 2.0   # frames sampled per second of clip

def num_frames_for_scenario(start_frame: int, end_frame: int, source_fps: float) -> int:
    duration_s = (end_frame - start_frame) / source_fps
    return max(MIN_FRAMES, round(duration_s * SAMPLE_FPS))


## 1. Load and clean the annotation CSV

In [ ]:

df = pd.read_csv(DATA_CSV)
print(f"{len(df)} raw scenarios, {df['video_path'].nunique()} unique source videos")

TAG_FAMILIES = ["pedestrian_behavior_tags", "vehicle_tags", "environment_tags", "archetypes"]

def split_tags(cell) -> list[str]:
    if pd.isna(cell) or str(cell).strip() == "":
        return []
    return [t.strip() for t in str(cell).split(",") if t.strip()]

for fam in TAG_FAMILIES:
    df[fam + "_list"] = df[fam].apply(split_tags)

# Curation per your Section II-B1: drop scenarios with no tags at all, and
# scenarios whose only tags don't describe a pedestrian-vehicle interaction.
# You'll need to supply the actual "non-interaction" tag set from your
# curation notes; the placeholder below only drops fully-empty scenarios.
NON_INTERACTION_ONLY_TAGS = set()  # TODO: fill from your curation log

def is_interaction_scenario(row) -> bool:
    all_tags = set(row["pedestrian_behavior_tags_list"]) | set(row["vehicle_tags_list"])
    if not all_tags:
        return False
    if NON_INTERACTION_ONLY_TAGS and all_tags <= NON_INTERACTION_ONLY_TAGS:
        return False
    return True

df_curated = df[df.apply(is_interaction_scenario, axis=1)].reset_index(drop=True)
print(f"{len(df_curated)} scenarios after curation filter "
      f"(paper reports 910 on the fully deduplicated/merged two-pass set — "
      f"this CSV is already single-pass merged, so treat this as your working set)")


## 2. Dataset statistics — reproduce Tables II-V (no VLM required)

These reproduce the archetype coverage, behavior/outcome profile, overlap, and
occlusion-subset tables directly from the CSV. **Table I (inter-annotator agreement)
cannot be reproduced here** — it requires the two independent pre-adjudication annotation
files (one per annotator, before merging), not this already-merged CSV. If you still have
the raw per-annotator exports from PedAnalyze, load those separately and reuse the
`kappa_table()` function stubbed at the bottom of this section.


In [ ]:

def tag_vocab_sizes(df):
    return {fam: sorted({t for row in df[fam + "_list"] for t in row}) for fam in TAG_FAMILIES}

vocab = tag_vocab_sizes(df_curated)
for fam, tags in vocab.items():
    print(f"{fam}: {len(tags)} unique tags")


In [ ]:

# ---- Table II: Archetype coverage + behavioral signature ------------------
def archetype_coverage(df, min_signature_frac=0.40, top_n_signature=2):
    rows = []
    for arche in vocab["archetypes"]:
        mask = df["archetypes_list"].apply(lambda tags: arche in tags)
        sub = df[mask]
        n = len(sub)
        if n == 0:
            continue
        behavior_counts = Counter(t for tags in sub["pedestrian_behavior_tags_list"] for t in tags)
        signature = [(t, c / n) for t, c in behavior_counts.most_common() if c / n >= min_signature_frac]
        signature = signature[:top_n_signature]
        rows.append({
            "archetype": arche.upper(),
            "scenarios": n,
            "signature": "; ".join(f"{t} ({frac:.1%})" for t, frac in signature),
        })
    return pd.DataFrame(rows).sort_values("scenarios", ascending=False).reset_index(drop=True)

table_ii = archetype_coverage(df_curated)
table_ii.to_csv(RESULTS_DIR / "table_ii_archetype_coverage.csv", index=False)
table_ii


In [ ]:

# ---- Table III: Archetype behavior/outcome profile -------------------------
PROFILE_LABELS = ["collision", "near-miss", "run-into-traffic", "ignore-traffic", "looking"]

def archetype_outcome_profile(df, labels=PROFILE_LABELS):
    rows = []
    for arche in vocab["archetypes"]:
        mask = df["archetypes_list"].apply(lambda tags: arche in tags)
        sub = df[mask]
        n = len(sub)
        if n == 0:
            continue
        row = {"archetype": arche.upper(), "scenarios": n}
        for label in labels:
            hit = sub["pedestrian_behavior_tags_list"].apply(lambda tags: label in tags).sum()
            row[label] = round(100 * hit / n, 1)
        rows.append(row)
    return pd.DataFrame(rows).sort_values("scenarios", ascending=False).reset_index(drop=True)

table_iii = archetype_outcome_profile(df_curated)
table_iii.to_csv(RESULTS_DIR / "table_iii_behavior_outcome_profile.csv", index=False)
table_iii


In [ ]:

# ---- Table IV: Selected archetype overlaps ---------------------------------
def archetype_overlaps(df, pairs=None, top_k=10):
    counts = {a: df["archetypes_list"].apply(lambda tags: a in tags).sum() for a in vocab["archetypes"]}
    if pairs is None:
        # default: report the top-k most frequent co-occurring pairs
        co = Counter()
        for tags in df["archetypes_list"]:
            uniq = sorted(set(tags))
            for i in range(len(uniq)):
                for j in range(i + 1, len(uniq)):
                    co[(uniq[i], uniq[j])] += 1
        pairs = [p for p, _ in co.most_common(top_k)]
    rows = []
    for a, b in pairs:
        shared = df.apply(lambda r: (a in r["archetypes_list"]) and (b in r["archetypes_list"]), axis=1).sum()
        rows.append({
            "archetype_a": a.upper(), "archetype_b": b.upper(), "shared": shared,
            "within_a_pct": round(100 * shared / counts[a], 1) if counts[a] else 0.0,
            "within_b_pct": round(100 * shared / counts[b], 1) if counts[b] else 0.0,
        })
    return pd.DataFrame(rows)

table_iv = archetype_overlaps(df_curated)
table_iv.to_csv(RESULTS_DIR / "table_iv_archetype_overlaps.csv", index=False)
table_iv


In [ ]:

# ---- Table V: Occlusion subset ---------------------------------------------
def occlusion_subset(df, anchor_tag="pop-out-occlusion"):
    mask = df["pedestrian_behavior_tags_list"].apply(lambda tags: anchor_tag in tags)
    sub = df[mask]
    n = len(sub)
    secondary_counts = Counter(t for tags in sub["pedestrian_behavior_tags_list"] for t in tags if t != anchor_tag)
    rows = [{"secondary_label": t, "scenarios": c, "within_subset_pct": round(100 * c / n, 2)}
            for t, c in secondary_counts.most_common()]
    return n, pd.DataFrame(rows)

subset_n, table_v = occlusion_subset(df_curated)
print(f"Occlusion subset size: {subset_n} scenarios")
table_v.to_csv(RESULTS_DIR / "table_v_occlusion_subset.csv", index=False)
table_v.head(10)


In [ ]:

# ---- Table I stub (needs raw dual-annotation files, not this CSV) ---------
def kappa_table(annotator_a_df, annotator_b_df, tag_families=TAG_FAMILIES):
    \"\"\"
    Cohen's kappa, Gwet's AC1, and PABAK per tag, computed over the two
    independent pre-adjudication annotation files. Both dataframes must have
    identical scenario IDs / frame intervals (this is why the paper fixes
    intervals after pass 1 before pass 2 runs).

    Not runnable in this notebook — supply the two raw per-annotator exports.
    \"\"\"
    all_tags = {fam: sorted({t for lst in annotator_a_df[fam + "_list"] for t in lst}
                             | {t for lst in annotator_b_df[fam + "_list"] for t in lst})
                for fam in tag_families}
    rows = []
    for fam, tags in all_tags.items():
        for tag in tags:
            a = annotator_a_df[fam + "_list"].apply(lambda l: tag in l).astype(int).values
            b = annotator_b_df[fam + "_list"].apply(lambda l: tag in l).astype(int).values
            n = len(a)
            po = (a == b).mean()
            pa1, pb1 = a.mean(), b.mean()
            pe = pa1 * pb1 + (1 - pa1) * (1 - pb1)
            kappa = (po - pe) / (1 - pe) if pe < 1 else float("nan")
            pi = a.mean() * 0.5 + b.mean() * 0.5  # mean prevalence, for AC1
            ac1_pe = 2 * pi * (1 - pi)
            ac1 = (po - ac1_pe) / (1 - ac1_pe) if ac1_pe < 1 else float("nan")
            pabak = 2 * po - 1
            rows.append({"family": fam, "tag": tag, "n": n, "percent_agreement": po,
                         "cohens_kappa": kappa, "gwets_ac1": ac1, "pabak": pabak})
    return pd.DataFrame(rows)

# Example (uncomment once you have the two raw exports):
# ann_a = pd.read_csv("annotator_a_raw.csv"); ann_a[...] = ...  # apply split_tags as above
# ann_b = pd.read_csv("annotator_b_raw.csv")
# kt = kappa_table(ann_a, ann_b)


## 3. Video acquisition and frame extraction

Downloads each of the 84 unique source videos once, then extracts per-scenario clips
and sampled frames using the interval + frame-sampling policy from Section IV-A / Fig. 1.


In [ ]:

def download_source_video(url: str) -> Path | None:
    vid_id = re.search(r"(?:v=|/)([0-9A-Za-z_-]{11})", url)
    vid_id = vid_id.group(1) if vid_id else re.sub(r"\\W+", "_", url)[-16:]
    out_path = VIDEO_DIR / f"{vid_id}.mp4"
    if out_path.exists():
        return out_path
    cmd = ["yt-dlp", "-f", "mp4", "-o", str(out_path), url]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"[WARN] failed to download {url}: {result.stderr[-300:]}")
        return None
    return out_path

def get_source_fps(video_path: Path) -> float:
    import cv2
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    cap.release()
    return fps

def extract_scenario_frames(video_path: Path, scenario_id, start_frame: int, end_frame: int) -> list[Path]:
    import cv2
    out_dir = FRAME_DIR / str(scenario_id)
    out_dir.mkdir(parents=True, exist_ok=True)
    existing = sorted(out_dir.glob("frame_*.jpg"))
    if existing:
        return existing

    fps = get_source_fps(video_path)
    n = num_frames_for_scenario(start_frame, end_frame, fps)
    frame_indices = np.linspace(start_frame, end_frame, n, dtype=int)

    cap = cv2.VideoCapture(str(video_path))
    saved = []
    for i, fidx in enumerate(frame_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(fidx))
        ok, frame = cap.read()
        if not ok:
            continue
        fp = out_dir / f"frame_{i:03d}.jpg"
        cv2.imwrite(str(fp), frame)
        saved.append(fp)
    cap.release()
    return saved

# Full pipeline (this will take a while for 84 videos — run once, it's cached to disk)
def build_frame_index(df, limit=None):
    index = {}
    unique_videos = df["video_path"].unique()
    if limit:
        unique_videos = unique_videos[:limit]
    video_cache = {}
    for url in tqdm(unique_videos, desc="videos"):
        video_cache[url] = download_source_video(url)

    for _, row in tqdm(df.iterrows(), total=len(df), desc="scenarios"):
        vp = video_cache.get(row["video_path"])
        if vp is None:
            continue
        frames = extract_scenario_frames(vp, row["id"], int(row["start_frame"]), int(row["end_frame"]))
        index[row["id"]] = [str(f) for f in frames]
    return index

# frame_index = build_frame_index(df_curated)          # full run
# frame_index = build_frame_index(df_curated, limit=5)  # smoke test first


## 4. Taxonomy definitions + prompt construction

Fill `TAXONOMY_DEFINITIONS` from your published documentation
(`pedanalyze.readthedocs.io/.../pedestrian-behavior-tags.html`) — the keys below are
auto-populated from the tag vocab found in the CSV so nothing is silently dropped, but
the definition strings are placeholders until you paste in the real ones.


In [ ]:

TAXONOMY_DEFINITIONS = {
    fam: {tag: "TODO: paste operational definition from PedAnalyze docs" for tag in tags}
    for fam, tags in vocab.items()
}

SYSTEM_PROMPT = \"\"\"You are annotating dashcam video clips of pedestrian-vehicle \
interactions using a fixed taxonomy. You will be given a sequence of frames sampled \
from one scenario, along with tag definitions for four label families: pedestrian \
behavior, vehicle response, environment condition, and archetype. Return ONLY a JSON \
object with exactly these four keys, each mapping to a list of applicable tag strings \
from the provided vocabulary. Do not invent tags outside the provided vocabulary. Do \
not add commentary outside the JSON object.\"\"\"

def build_taxonomy_block(definitions=TAXONOMY_DEFINITIONS) -> str:
    lines = []
    for fam, tag_defs in definitions.items():
        fam_label = fam.replace("_tags", "").replace("_", " ")
        lines.append(f"### {fam_label}")
        for tag, definition in tag_defs.items():
            lines.append(f"- {tag}: {definition}")
    return "\n".join(lines)

TAXONOMY_BLOCK = build_taxonomy_block()

USER_INSTRUCTION_TEMPLATE = \"\"\"Here are {n_frames} frames sampled uniformly from a \
pre-event dashcam clip. Identify every applicable tag from each of the four families \
below. Output strict JSON: {{"pedestrian_behavior_tags": [...], "vehicle_tags": [...], \
"environment_tags": [...], "archetypes": [...]}}

TAXONOMY:
{taxonomy_block}
\"\"\"

def build_prompt(n_frames: int) -> tuple[str, str]:
    return SYSTEM_PROMPT, USER_INSTRUCTION_TEMPLATE.format(n_frames=n_frames, taxonomy_block=TAXONOMY_BLOCK)


## 5. Model wrappers

One thin wrapper per model, all returning a raw string (the model's JSON-or-not output).
Parsing/fuzzy-matching is handled separately in Section 6 so a model's response format
quirks don't leak into scoring.


In [ ]:

import base64

def _encode_image(path: str) -> str:
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

# ---- GPT-4o / GPT-4o-mini (OpenAI) -----------------------------------------
def call_gpt4o(frame_paths: list[str], model="gpt-4o") -> str:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    system_prompt, user_text = build_prompt(len(frame_paths))
    content = [{"type": "text", "text": user_text}]
    for fp in frame_paths:
        content.append({"type": "image_url",
                         "image_url": {"url": f"data:image/jpeg;base64,{_encode_image(fp)}"}})
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": system_prompt},
                   {"role": "user", "content": content}],
        temperature=0,
    )
    return resp.choices[0].message.content

# ---- Claude (Anthropic) ----------------------------------------------------
def call_claude(frame_paths: list[str], model="claude-sonnet-4-6") -> str:
    import anthropic
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    system_prompt, user_text = build_prompt(len(frame_paths))
    content = [{"type": "text", "text": user_text}]
    for fp in frame_paths:
        content.append({"type": "image", "source": {"type": "base64", "media_type": "image/jpeg",
                                                      "data": _encode_image(fp)}})
    resp = client.messages.create(
        model=model, max_tokens=1024, temperature=0,
        system=system_prompt,
        messages=[{"role": "user", "content": content}],
    )
    return resp.content[0].text

# ---- Gemini -----------------------------------------------------------------
def call_gemini(frame_paths: list[str], model="gemini-2.5-flash") -> str:
    import google.generativeai as genai
    genai.configure(api_key=GOOGLE_API_KEY)
    system_prompt, user_text = build_prompt(len(frame_paths))
    gmodel = genai.GenerativeModel(model, system_instruction=system_prompt)
    parts = [user_text] + [{"mime_type": "image/jpeg", "data": open(fp, "rb").read()} for fp in frame_paths]
    resp = gmodel.generate_content(parts, generation_config={"temperature": 0})
    return resp.text

# ---- Qwen3-VL-32B: local / Modal (see Section 8 for the Modal deployment) --
def call_qwen3_vl(frame_paths: list[str], endpoint_url: str) -> str:
    \"\"\"Calls a deployed Modal endpoint (see modal_app.py in Section 8)
    rather than loading the model in this process.\"\"\"
    import requests
    system_prompt, user_text = build_prompt(len(frame_paths))
    payload = {"system_prompt": system_prompt, "user_text": user_text,
               "frames_b64": [_encode_image(fp) for fp in frame_paths]}
    resp = requests.post(endpoint_url, json=payload, timeout=120)
    resp.raise_for_status()
    return resp.json()["text"]

MODEL_REGISTRY = {
    "gpt-4o":            lambda frames: call_gpt4o(frames, "gpt-4o"),
    "gpt-4o-mini":       lambda frames: call_gpt4o(frames, "gpt-4o-mini"),
    "claude-sonnet-4-6": lambda frames: call_claude(frames, "claude-sonnet-4-6"),
    "gemini-2.5-flash":  lambda frames: call_gemini(frames, "gemini-2.5-flash"),
    # "qwen3-vl-32b":    lambda frames: call_qwen3_vl(frames, QWEN_ENDPOINT_URL),
}


## 6. Response parser + fuzzy matcher

In [ ]:

def parse_model_response(raw_text: str, vocab=vocab, fuzzy_threshold=85):
    \"\"\"Returns (parsed_dict, status) where status in {"ok", "fuzzy_matched", "invalid_json"}.
    Every returned tag is one of: a valid vocab tag, or dropped (logged as unmatched)
    if it doesn't clear the fuzzy-match threshold against the family vocabulary.\"\"\"
    text = raw_text.strip()
    text = re.sub(r"^```(?:json)?|```$", "", text, flags=re.MULTILINE).strip()
    try:
        obj = json.loads(text)
    except json.JSONDecodeError:
        return {fam: [] for fam in vocab}, "invalid_json"

    result, any_fuzzy, unmatched = {}, False, []
    for fam, tags in vocab.items():
        raw_list = obj.get(fam, [])
        if not isinstance(raw_list, list):
            raw_list = []
        matched = []
        for t in raw_list:
            t_norm = str(t).strip().lower()
            if t_norm in {v.lower() for v in tags}:
                matched.append(next(v for v in tags if v.lower() == t_norm))
                continue
            best = rf_process.extractOne(t_norm, tags, scorer=fuzz.token_sort_ratio)
            if best and best[1] >= fuzzy_threshold:
                matched.append(best[0])
                any_fuzzy = True
            else:
                unmatched.append((fam, t))
        result[fam] = sorted(set(matched))
    status = "fuzzy_matched" if any_fuzzy else "ok"
    if unmatched:
        result["_unmatched"] = unmatched
    return result, status


## 7. Task 1 — DOPe-Understand: zero-shot multi-label recognition

Runs every registered model over every scenario, parses responses, and scores
precision/recall/F1/Jaccard per family (macro- and micro-averaged), matching
Section IV-C of your draft.


In [ ]:

def run_understand_eval(df, frame_index, models=MODEL_REGISTRY, max_retries=2, sleep_s=1.0):
    records = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="scenarios"):
        frames = frame_index.get(row["id"])
        if not frames:
            continue
        gold = {fam: set(row[fam + "_list"]) for fam in TAG_FAMILIES}
        for model_name, call_fn in models.items():
            raw, status = None, "error"
            for attempt in range(max_retries):
                try:
                    raw = call_fn(frames)
                    break
                except Exception as e:
                    time.sleep(sleep_s * (attempt + 1))
            if raw is None:
                records.append({"scenario_id": row["id"], "model": model_name, "status": "api_error"})
                continue
            parsed, status = parse_model_response(raw)
            rec = {"scenario_id": row["id"], "model": model_name, "status": status,
                   "raw_response": raw}
            for fam in TAG_FAMILIES:
                rec[f"gold_{fam}"] = sorted(gold[fam])
                rec[f"pred_{fam}"] = parsed.get(fam, [])
            records.append(rec)
    return pd.DataFrame(records)

# results_understand = run_understand_eval(df_curated, frame_index)
# results_understand.to_json(RESULTS_DIR / "understand_raw.jsonl", orient="records", lines=True)


In [ ]:

def prf_jaccard(gold: set, pred: set):
    tp = len(gold & pred)
    fp = len(pred - gold)
    fn = len(gold - pred)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    union = len(gold | pred)
    jaccard = tp / union if union else 1.0
    return precision, recall, f1, jaccard

def score_understand(results_df, vocab=vocab):
    \"\"\"Micro (scenario-level) and macro (per-tag) metrics per family per model.\"\"\"
    rows = []
    for model_name, sub in results_df.groupby("model"):
        for fam in TAG_FAMILIES:
            # micro: pool tp/fp/fn across all scenarios
            tp = fp = fn = 0
            jaccards = []
            for _, r in sub.iterrows():
                g, p = set(r[f"gold_{fam}"]), set(r[f"pred_{fam}"])
                tp += len(g & p); fp += len(p - g); fn += len(g - p)
                _, _, _, j = prf_jaccard(g, p)
                jaccards.append(j)
            micro_p = tp / (tp + fp) if (tp + fp) else 0.0
            micro_r = tp / (tp + fn) if (tp + fn) else 0.0
            micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) else 0.0

            # macro: per-tag F1 then average, only over tags with >=1 positive in gold
            per_tag_f1 = []
            for tag in vocab[fam]:
                g = sub[f"gold_{fam}"].apply(lambda l: tag in l)
                if g.sum() == 0:
                    continue  # per spec: exclude tags with no positive support from macro avg
                p = sub[f"pred_{fam}"].apply(lambda l: tag in l)
                tpv = (g & p).sum(); fpv = (p & ~g).sum(); fnv = (g & ~p).sum()
                pr = tpv / (tpv + fpv) if (tpv + fpv) else 0.0
                rc = tpv / (tpv + fnv) if (tpv + fnv) else 0.0
                f1v = 2 * pr * rc / (pr + rc) if (pr + rc) else 0.0
                per_tag_f1.append(f1v)
            macro_f1 = float(np.mean(per_tag_f1)) if per_tag_f1 else float("nan")

            rows.append({
                "model": model_name, "family": fam,
                "micro_precision": round(micro_p, 3), "micro_recall": round(micro_r, 3),
                "micro_f1": round(micro_f1, 3), "macro_f1": round(macro_f1, 3),
                "mean_jaccard": round(float(np.mean(jaccards)), 3),
                "n_scenarios": len(sub), "n_tags_with_support": len(per_tag_f1),
            })
    return pd.DataFrame(rows)

# scores_understand = score_understand(results_understand)
# scores_understand.to_csv(RESULTS_DIR / "table_main_understand.csv", index=False)


## 8. Qwen3-VL-32B via Modal (the only GPU-bound step)

Deploy once as a Modal app, then `call_qwen3_vl()` in Section 5 just POSTs to it —
keeps everything else in this notebook CPU-only.

### GPU sizing detail
- **bf16 weights**: 32B params x 2 bytes ≈ 64 GB, plus KV-cache and vision-tower
  activations for ~8-16 frames per call → budget 70-78 GB actually used. A single
  **A100-80GB or H100-80GB** covers this with headroom; a 40GB card will OOM on bf16.
- **4-bit quantized (bitsandbytes NF4 or AWQ)**: weights drop to ~18-20 GB; with
  activations, **20-24 GB** total → a single **A100-40GB, L40S-48GB, or RTX 6000 Ada**
  is enough. Expect a modest (typically 2-5 point) drop in label F1 vs bf16 — worth
  running a small paired comparison on ~50 scenarios before committing to 4-bit for
  the full sweep.
- **Batching**: process one scenario (its full frame set) per forward pass rather than
  batching scenarios together — variable frame counts per scenario make padding
  wasteful, and dashcam frames are large enough that batching scenarios rarely helps
  more than modest concurrency at the request level.
- **Throughput estimate**: ~910 scenarios x ~8 frames average ≈ 7,300 frames. At
  roughly 3-6 s/scenario on an A100-80GB (dominated by vision-tower + prefill, not
  decode, since output is short JSON), a full pass is **~45-90 minutes** of GPU time.
  Budget 3x that if you're also running the Compose and Early proxy sweeps below.


In [ ]:

MODAL_APP_STUB = '''
# modal_app.py — deploy with: modal deploy modal_app.py
import modal

app = modal.App("dope-qwen3-vl")

image = (
    modal.Image.debian_slim(python_version="3.11")
    .pip_install("torch", "transformers>=4.45", "accelerate", "qwen-vl-utils", "pillow", "fastapi")
)

MODEL_ID = "Qwen/Qwen3-VL-32B-Instruct"  # verify the exact release tag you want

@app.cls(gpu="A100-80GB", image=image, timeout=600, scaledown_window=300)
class Qwen3VL:
    @modal.enter()
    def load(self):
        import torch
        from transformers import AutoModelForVision2Seq, AutoProcessor
        self.model = AutoModelForVision2Seq.from_pretrained(
            MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
        )
        self.processor = AutoProcessor.from_pretrained(MODEL_ID)

    @modal.method()
    def generate(self, system_prompt: str, user_text: str, frames_b64: list[str]) -> str:
        import base64, io
        from PIL import Image
        images = [Image.open(io.BytesIO(base64.b64decode(b))) for b in frames_b64]
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": [{"type": "text", "text": user_text}]
                                          + [{"type": "image"} for _ in images]},
        ]
        prompt = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.processor(text=prompt, images=images, return_tensors="pt").to(self.model.device)
        out = self.model.generate(**inputs, max_new_tokens=512, do_sample=False)
        return self.processor.decode(out[0], skip_special_tokens=True)

@app.function(image=image)
@modal.fastapi_endpoint(method="POST")
def endpoint(payload: dict):
    qwen = Qwen3VL()
    text = qwen.generate.remote(payload["system_prompt"], payload["user_text"], payload["frames_b64"])
    return {"text": text}
'''

with open("modal_app.py", "w") as f:
    f.write(MODAL_APP_STUB)
print("Wrote modal_app.py — deploy with: modal deploy modal_app.py")
print("For the 4-bit variant, swap gpu='A100-80GB' -> gpu='A100-40GB' and add")
print("BitsAndBytesConfig(load_in_4bit=True) to from_pretrained().")


## 9. DOPe-Compose-lite — compositional generalization proxy

Defines "seen" vs "unseen" **pairs of behavior tags** (not full risk-factor
combinations, since we don't have a validated risk-factor ontology beyond the raw
tags) — hold out a set of tag *pairs* from training-style exposure and compare
recognition accuracy on scenarios that contain an unseen pair vs scenarios that only
contain seen pairs. This is a proxy for the master-spec's compositional-risk claim,
scoped to what's actually annotated.


In [ ]:

def build_seen_unseen_split(df, family="pedestrian_behavior_tags_list", n_holdout_pairs=15, seed=RANDOM_SEED):
    rng = random.Random(seed)
    all_pairs = set()
    for tags in df[family]:
        uniq = sorted(set(tags))
        for i in range(len(uniq)):
            for j in range(i + 1, len(uniq)):
                all_pairs.add((uniq[i], uniq[j]))
    holdout_pairs = set(rng.sample(sorted(all_pairs), min(n_holdout_pairs, len(all_pairs))))

    def contains_holdout(tags):
        uniq = sorted(set(tags))
        for i in range(len(uniq)):
            for j in range(i + 1, len(uniq)):
                if (uniq[i], uniq[j]) in holdout_pairs:
                    return True
        return False

    is_unseen = df[family].apply(contains_holdout)
    return df[~is_unseen].reset_index(drop=True), df[is_unseen].reset_index(drop=True), holdout_pairs

seen_split, unseen_split, holdout_pairs = build_seen_unseen_split(df_curated)
print(f"seen: {len(seen_split)} scenarios, unseen (contains a held-out tag pair): {len(unseen_split)} scenarios")
print(f"held-out pairs (sample): {list(holdout_pairs)[:5]}")

# Run Section 7's eval separately on `seen_split` and `unseen_split` (same frame_index),
# then compare per-model macro_f1 for pedestrian_behavior_tags:
#   delta_comp = macro_f1(seen) - macro_f1(unseen)


## 10. DOPe-Early-lite — context-truncation degradation proxy

For each scenario, builds progressively shorter clips by truncating from the *end* of
the annotated interval (i.e., giving the model less of the run-up to whatever the
annotated interval captures). This measures how recognition degrades with less
temporal context — a genuine and useful ablation — but it is **not** a true
early-warning curve, because the CSV has no independent timestamp for "the critical
event" inside the interval to measure lead time against. Frame the results as a
context-sufficiency ablation, not an early-warning result, in the paper.


In [ ]:

TRUNCATION_FRACTIONS = [1.0, 0.75, 0.5, 0.25]

def build_truncated_frame_index(df, frame_index, fractions=TRUNCATION_FRACTIONS):
    truncated = {frac: {} for frac in fractions}
    for _, row in df.iterrows():
        frames = frame_index.get(row["id"])
        if not frames:
            continue
        for frac in fractions:
            keep_n = max(1, round(len(frames) * frac))
            truncated[frac][row["id"]] = frames[:keep_n]
    return truncated

# truncated_index = build_truncated_frame_index(df_curated, frame_index)
# for frac, idx in truncated_index.items():
#     res = run_understand_eval(df_curated, idx)
#     res["truncation_fraction"] = frac
#     res.to_json(RESULTS_DIR / f"early_lite_frac_{frac}.jsonl", orient="records", lines=True)
#
# Then plot macro_f1(pedestrian_behavior_tags) vs truncation_fraction per model —
# this is your "context sufficiency" curve, the honest stand-in for Fig. "Early warning curve".


## 11. Failure taxonomy (semi-automated F1-F7 flags)

In [ ]:

def flag_failure_types(row):
    \"\"\"Rule-based first pass at your F1-F7 taxonomy from a single scenario's
    per-family gold/pred sets. This is a coarse heuristic meant to triage
    scenarios for manual qualitative review, not a substitute for it.\"\"\"
    flags = []
    gold_beh, pred_beh = set(row["gold_pedestrian_behavior_tags"]), set(row["pred_pedestrian_behavior_tags"])
    gold_env, pred_env = set(row["gold_environment_tags"]), set(row["pred_environment_tags"])
    gold_veh, pred_veh = set(row["gold_vehicle_tags"]), set(row["pred_vehicle_tags"])
    gold_arc, pred_arc = set(row["gold_archetypes"]), set(row["pred_archetypes"])

    if gold_beh - pred_beh:
        flags.append("F1_behavior_miss")
    if not (gold_beh - pred_beh) and (gold_env - pred_env):
        flags.append("F2_context_miss")
    if not (gold_beh - pred_beh) and not (gold_veh - pred_veh) and (gold_arc != pred_arc) and gold_arc and pred_arc:
        flags.append("F3_interaction_miss")  # both agents' surface tags right, archetype composition wrong
    if "collision" in gold_beh and "collision" not in pred_beh:
        flags.append("F6_outcome_confusion")
    if len(gold_arc - pred_arc) >= 1 and len(gold_beh - pred_beh) == 0:
        flags.append("F5_composition_failure_candidate")  # got behaviors right, missed the composed archetype
    return flags

# results_understand["failure_flags"] = results_understand.apply(flag_failure_types, axis=1)
# results_understand.explode("failure_flags")["failure_flags"].value_counts()


## 12. Results aggregation + plotting

In [ ]:

import matplotlib.pyplot as plt

def plot_main_results(scores_df, metric="macro_f1"):
    pivot = scores_df.pivot(index="model", columns="family", values=metric)
    ax = pivot.plot(kind="bar", figsize=(9, 5))
    ax.set_ylabel(metric)
    ax.set_title(f"DOPe-Understand: {metric} by model and tag family")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"main_results_{metric}.png", dpi=150)
    plt.show()

# plot_main_results(scores_understand)


## 13. Limitations and what's needed to reach the master-spec scope

- **Table I (κ)**: needs your two raw per-annotator PedAnalyze exports (pre-merge).
  If you still have them, load into `kappa_table()` in Section 2 — ~30 min of work.
- **True DOPe-Risk**: needs per-scenario distance/relative-velocity/TTC. Cheapest
  path: run a monocular tracker (e.g., ByteTrack + a simple pinhole distance proxy)
  over the 910 clips and validate a subsample by hand — probably a 1-2 week job, not
  a notebook cell, and should be scoped as its own annotation-quality section in the
  paper if you pursue it.
- **True DOPe-Intervene**: needs (agent, action, timing) counterfactual labels per
  scenario. This is a new annotation pass — realistically 900 scenarios x ~2 min each
  ≈ 30 annotator-hours for a single pass, doubled if you want the same two-pass +
  adjudication protocol as the rest of DOPe.
- **True DOPe-Early**: needs a labeled "critical event" timestamp distinct from the
  scenario interval boundaries, so lead time can be measured. Without it, Section 10's
  truncation proxy is the most defensible thing to report.
- Given your stated preference for benchmark/dataset-paper formats and avoiding scope
  creep, the pragmatic ICRA-2027 story is: ship **DOPe-Understand + dataset analysis +
  Compose-lite + the context-sufficiency ablation** now, and flag Risk/Intervene/true-Early
  as a stated future-work extension requiring a second annotation pass — rather than
  quietly presenting the truncation proxy as if it were early-warning, which a reviewer
  who reads Section II-B closely will catch.
